# ETF Signals 2.0 — análise interativa em Google Colab
Carregue o ficheiro `etf_sinais_v2.zip` e execute as células por ordem. Os gráficos e os CSV são mostrados diretamente no notebook, sem servidor público.

In [ ]:
!pip -q install yfinance plotly ipywidgets pandas numpy

In [ ]:
from google.colab import files
uploaded=files.upload()
import io, zipfile, sys
filename=next(k for k in uploaded if k.endswith(".zip"))
zipfile.ZipFile(io.BytesIO(uploaded[filename])).extractall("/content")
sys.path.insert(0,"/content/etf_sinais_v2")

In [ ]:
import pandas as pd, numpy as np, plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display,clear_output
import ipywidgets as wd
from engine import CATALOG, fetch, calc_signal, portfolio
import datetime
selection=wd.SelectMultiple(options=[(f"{t}: {v[1]}",t) for t,v in CATALOG.items()],value=("DIA","RSP","QQQ"),description="ETFs",rows=12,layout=wd.Layout(width="440px"))
capital=wd.FloatText(value=10000,description="Capital USD")
start=wd.Text(value="2016-01-01",description="Início")
fee=wd.FloatText(value=10,description="Comissão bp")
slip=wd.FloatText(value=5,description="Slippage bp")
run=wd.Button(description="Analisar",button_style="success")
out=wd.Output()
display(wd.VBox([selection,capital,start,fee,slip,run,out]))

In [ ]:
def analyse(_):
  with out:
    clear_output(wait=True)
    selected=list(selection.value)
    if not selected:
      print("Selecionar pelo menos um ETF");return
    data=fetch(selected,str(pd.Timestamp(start.value)-pd.Timedelta(days=900)))
    missing=[t for t in selected if t not in data]
    if missing:print("Sem preços para",missing);return
    signals={t:calc_signal(t,data) for t in selected}
    weights={t:1/len(selected) for t in selected}
    hist,orders,m,b=portfolio(signals,data,weights,capital.value,fee.value,slip.value,start.value)
    latest=pd.DataFrame([{'ETF':t,'Fecho':str(x.index[-1].date()),'Preço':round(x.Close.iloc[-1],2),'Score':round(x.score.iloc[-1],1),'Gate':bool(x.gate.iloc[-1]),'Alvo':int(x.position_target.iloc[-1]),'Sinal':x.signal.iloc[-1]} for t,x in signals.items()])
    display(latest)
    print('Carteira:',m,'  | SPY buy and hold:',b)
    fig=go.Figure()
    fig.add_trace(go.Scatter(x=hist.index,y=hist.nav,name='Estratégia'))
    fig.add_trace(go.Scatter(x=hist.index,y=hist.bh_spy,name='SPY Buy & Hold'))
    fig.update_layout(title='Capital da carteira (USD)',height=420)
    fig.show()
    for t,x in signals.items():
      x=x.tail(360)
      price=data[t].reindex(x.index)
      fig=make_subplots(rows=2,cols=1,shared_xaxes=True,row_heights=[.72,.28],vertical_spacing=.07)
      fig.add_trace(go.Candlestick(x=price.index,open=price.Open,high=price.High,low=price.Low,close=price.Close,name=t),row=1,col=1)
      for c in ['sma20','sma50','sma200']:
        fig.add_trace(go.Scatter(x=x.index,y=x[c],name=c,mode='lines'),row=1,col=1)
      if len(orders):
        oo=orders[orders.ticker.eq(t)].copy();oo['execution_date']=pd.to_datetime(oo.execution_date)
        for side,shape,color in [('COMPRA','triangle-up','green'),('VENDA','triangle-down','red')]:
          pp=oo[oo.side.eq(side)]
          fig.add_trace(go.Scatter(x=pp.execution_date,y=pp.price_open,mode='markers',name=side,marker={'symbol':shape,'size':12,'color':color}),row=1,col=1)
      fig.add_trace(go.Scatter(x=x.index,y=x.score,name='Score'),row=2,col=1)
      fig.add_hline(y=70,line_dash='dash',row=2,col=1);fig.add_hline(y=40,line_dash='dash',row=2,col=1)
      fig.update_yaxes(range=[0,100],row=2,col=1)
      fig.update_layout(title=f'{t}: Velas diárias, score e ordens executadas',height=680,xaxis_rangeslider_visible=False)
      fig.show()
      x.to_csv(f'/content/sinais_{t}_v2.csv',index_label='data')
    latest.to_csv('/content/resumo_v2.csv',index=False)
    hist.to_csv('/content/carteira_v2.csv',index_label='data')
    orders.to_csv('/content/operacoes_v2.csv',index=False)
    print('CSV disponíveis em /content/ no painel Ficheiros do Colab.')
run.on_click(analyse)
print('Configuração concluída: clique em Analisar acima.')